In [1]:
import json
import os
import pickle
import sys

import datetime
import httpx
import pandas as pd
import requests
import time
import yaml
from dotenv import load_dotenv
from tqdm import tqdm


# Determine the project root directory for relative imports
try:
    # This will work in scripts where __file__ is defined
    current_dir = os.path.dirname(os.path.abspath(__file__))
    # Assuming "src" is parallel to the script folder
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
except NameError:
    # In notebooks __file__ is not defined: assume we're in notebooks/riziv_dataset/
    project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)



In [2]:
load_dotenv(os.path.join(project_root, ".env"))
response_gpu_temp = httpx.get(f"{os.getenv('NGROK_URL')}/health/gpu")

response_gpu_temp

<Response [200 OK]>

In [3]:
int(response_gpu_temp.json()["temperature"])

51

In [ ]:
###############
# Load BSARD dataset main table (lean)
###############

# Define the path to the BSARD dataset files
BSARD_data_path = os.path.join(project_root, "data", "BSARD_dataset")

bsard_corpus_lean = pd.read_csv(os.path.join(BSARD_data_path, 'intermediate', "bsard_corpus_lean_V2.csv"))
bsard_corpus_lean.drop(["Unnamed: 0"], inplace=True, axis=1)
bsard_corpus_lean


###############
# Load dotenv
###############

load_dotenv(os.path.join(project_root, ".env"))


###############
# Load prompts
###############

prompts_path = os.path.join(project_root, "prompts", "BSARD_keyword_extraction.yaml")

with open(prompts_path, 'r') as f:
    prompts = yaml.safe_load(f)


###############
# Load/create keywords dict and determine targets
###############

if "keywords_dict" not in globals():

    try: # Try to load dict
        keywords_dict_path = os.path.join(BSARD_data_path, 'intermediate', 'keyword_extraction.pkl')

        with open(keywords_dict_path, 'rb') as f:
            keywords_dict = pickle.load(f)

    except: # Create new one
        keywords_dict = dict.fromkeys(bsard_corpus_lean['article_code'], None)

else:

    pass

keyword_dict_unscanned = pd.Series([k for k,v in keywords_dict.items() if v == None])


###############
# Launch (Main) Loop
###############

save_counter = 0
error_counter = 0
gpu_temp_counter = 0

for article in tqdm(list(keyword_dict_unscanned), desc="Scanning articles", miniters=50):

    if save_counter < 25: # Add 1 to the counter
        save_counter += 1

    elif save_counter >= 25: # Save results every 10 iterations and reset counter
        save_counter = 0
        with open(os.path.join(BSARD_data_path, 'intermediate', "keyword_extraction.pkl"), 'wb') as f:
            pickle.dump(keywords_dict, f)
        print(f"Checkpoint saved! - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} \n")

    try:

        ##########
        # Sample Random Article and Build prompt
        ##########

        # Sample random article witthin un-scanned ones
        random_article = list(keyword_dict_unscanned.sample())[0]

        # Extract and format relevant info for prompt building
        tmp_df = bsard_corpus_lean[bsard_corpus_lean["article_code"]==random_article]
        parsed_act = list(tmp_df["parsed_act"])[0]
        parsed_book = list(tmp_df["parsed_book"])[0]
        parsed_title = list(tmp_df["parsed_title"])[0]
        parsed_article = list(tmp_df["parsed_art"])[0]
        article_text = list(tmp_df["article"])[0]

        context_act = f"LAW: {parsed_act}"
        context_subdivisions = f"SUBDIVISIONS: book - {parsed_act}, tile - {parsed_title}, article - {parsed_article}"
        context_main_text = f"MAIN TEXT: {article_text}"

        # Configure prompt
        joint_prompt = str(prompts["keyword-extraction_A"] + "\n" + context_act + "\n" + context_subdivisions + "\n" + context_main_text)

        #print("\n" + joint_prompt + "\n")


        ##########
        # LLM Call (API)
        ##########

        NGROK_URL = os.getenv("NGROK_URL")
        API_TOKEN = os.getenv("API_KEY")

        output_schema = {
            "type": "object",
            "properties": {

                "key_concept_1": { "type": "string" },
                "justification_key_concept_1": { "type": "string" },

                "key_concept_2": { "type": "string" },
                "justification_key_concept_2": { "type": "string" },

                "key_concept_3": { "type": "string" },
                "justification_key_concept_3": { "type": "string" },

                "key_concept_4": { "type": "string" },
                "justification_key_concept_4": { "type": "string" },

            },
            "required": [
                "key_concept_1", "justification_key_concept_1",
                "key_concept_2", "justification_key_concept_2", 
                "key_concept_3", "justification_key_concept_3", 
                "key_concept_4", "justification_key_concept_4", 
                         ]
        }

        response = requests.post(
            f"{NGROK_URL}/generate",
            headers={
                "Authorization": f"Bearer {API_TOKEN}",
            },
            json={
                "model": "gemma3:12b-it-fp16",
                "prompt": joint_prompt,
                "output_format": output_schema
            }
        )
        #print("\n" + str(response.json()) + "\n")

        ##########
        # Store result in dictionary
        ##########

        response_main = json.loads(response.json()['response'])
        keywords_dict[random_article] = response_main

        #print(response_main)

        ##########
        # GPU safety
        ##########

        response_gpu_temp = httpx.get(f"{NGROK_URL}/health/gpu")
        if int(response_gpu_temp.json()["temperature"]) > 83:
            print("Detected GPU temperature upon request completion above defined threshold (83)...")
            print(f"Starting safety cooldown break (180 sec.) - ({datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')})")
            time.sleep(180)

        gpu_temp_counter += 1
        if gpu_temp_counter == 50:

            response_gpu_temp = httpx.get(f"{NGROK_URL}/health/gpu")
            print(str(response.status_code) + " - GPU Temperature upon request completion: " + str(response_gpu_temp.json()["temperature"]) + "\n")

            gpu_temp_counter = 0

    except:

        print("Unknown error encountered....")
        error_counter += 1

        if error_counter == 5:
            print("Error counter reached maximum threshold for a single run... INTERRUPTING OPERATION")
            break
        
        continue


Scanning articles:   6%|▌         | 25/425 [04:44<1:43:56, 15.59s/it]

Checkpoint saved! - 2025-06-28 22:03:13 



Scanning articles:  12%|█▏        | 50/425 [10:57<1:32:19, 14.77s/it]

200 - GPU Temperature upon request completion: 72



Scanning articles:  12%|█▏        | 51/425 [11:09<1:26:31, 13.88s/it]

Checkpoint saved! - 2025-06-28 22:09:38 



Scanning articles:  17%|█▋        | 72/425 [16:28<1:30:08, 15.32s/it]

In [ ]:
with open(os.path.join(BSARD_data_path, 'intermediate', "keyword_extraction.pkl"), 'wb') as f:
    pickle.dump(keywords_dict, f)